# 🚗 Indian ANPR System - Free Google Colab Backend Server
**Hardware:** Google Cloud (12.7 GB RAM + Free GPU/CPU)  
**Status:** 100% Free, Zero Out-Of-Memory Crashes  
---
### Instructions:
1. *(Optional)* Paste your **Ngrok Authtoken** & **Permanent Domain** below to get a permanent URL that NEVER changes.
2. Click the **Play (Run)** button.
3. Wait until the backend is verified online.

In [ ]:
# =============================================================
# ⚙️ CONFIGURATION
# =============================================================
REPO_URL = "https://github.com/amithneranjane/ANPR_FINAL.git"

# OPTIONAL: For a 100% Permanent URL that NEVER changes (Free from ngrok.com):
# Leave blank ("") to use automatic Cloudflare tunnel instead.
NGROK_AUTHTOKEN = ""
NGROK_DOMAIN = ""
# =============================================================

import os, subprocess, time, re, sys, urllib.request, json

!rm -rf anpr
print("📥 Cloning latest repository from GitHub...")
!git clone $REPO_URL anpr
%cd anpr

print("📦 Installing Python dependencies (takes ~1 minute)...")
!pip install -q -r requirements.txt pyngrok

# Step 2: Download Cloudflare Tunnel binary
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

# Step 3: Start Flask Server with logging
print("🚀 Launching Flask ANPR Backend Server...")
flask_log = open("flask.log", "w")
flask_proc = subprocess.Popen(["python", "app.py"], stdout=flask_log, stderr=subprocess.STDOUT)

# Step 4: Health check loop (wait until YOLO & EasyOCR are fully loaded)
print("⏳ Waiting for YOLOv8 & EasyOCR models to initialize on port 5000...")
server_ready = False
for i in range(45):
    if flask_proc.poll() is not None:
        print("❌ Flask server process terminated unexpectedly. Log output:")
        with open("flask.log", "r") as f:
            print(f.read())
        sys.exit(1)
    try:
        req = urllib.request.Request("http://127.0.0.1:5000/health")
        with urllib.request.urlopen(req, timeout=2) as resp:
            if resp.status == 200:
                server_ready = True
                break
    except Exception:
        time.sleep(1.5)
        if (i + 1) % 5 == 0:
            print(f"   ... still loading models ({i * 1.5:.0f}s elapsed)")

if not server_ready:
    print("❌ Flask server did not respond in time. Checking flask.log:")
    with open("flask.log", "r") as f:
        print(f.read())
    sys.exit(1)

print("✅ Flask Backend is READY and responding on port 5000!")

# Step 5: Start Tunnel (Ngrok Permanent Domain or Cloudflare)
public_url = None

if NGROK_AUTHTOKEN.strip() and NGROK_DOMAIN.strip():
    print("🌐 Connecting to your PERMANENT Ngrok Domain...")
    from pyngrok import ngrok
    ngrok.kill()
    ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())
    tunnel = ngrok.connect(5000, domain=NGROK_DOMAIN.strip())
    public_url = tunnel.public_url
    if public_url.startswith("http://"):
        public_url = public_url.replace("http://", "https://")
    print("=" * 65)
    print("🎉 YOUR PERMANENT ANPR BACKEND SERVER IS ONLINE!")
    print(f"🔗 PERMANENT URL: {public_url}")
    print("=" * 65)
    print("✨ This URL NEVER changes! Render will always connect automatically.")
else:
    print("🌐 Opening Cloudflare HTTPS Tunnel...")
    cf_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:5000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
    for _ in range(40):
        line = cf_proc.stdout.readline()
        if not line: continue
        m = url_pattern.search(line)
        if m:
            public_url = m.group(0)
            break
        time.sleep(0.5)
    print("=" * 65)
    print("🎉 YOUR ANPR BACKEND SERVER IS VERIFIED ONLINE!")
    print(f"🔗 BACKEND URL: {public_url}")
    print("=" * 65)
